# 🎤 Kani TTS Basque (Euskera) Finetuning

This notebook provides a complete pipeline for finetuning Kani TTS on Basque language audio data.

**What this notebook does:**
- Sets up the environment
- Prepares your Basque audio dataset
- Finetunes the Kani TTS model
- Saves the trained model
- Tests the model with Basque text

**Requirements:**
- Google Colab with GPU (T4 or better)
- Basque audio dataset (WAV files + transcripts)
- ~2-5 hours for training (depending on dataset size)

---

## 1. Setup Environment

Install required packages:

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets librosa pandas soundfile
!pip install -q "nemo_toolkit[tts]"
!pip install -q accelerate tensorboard safetensors

print("✅ All packages installed successfully!")

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU found. Training will be very slow!")
    print("   Go to Runtime -> Change runtime type -> Select GPU")

## 2. Prepare Dataset

### Option A: Upload Your Dataset

Upload your Basque audio dataset:
1. Create a folder structure:
   ```
   euskera_data/
   ├── audio/
   │   ├── sample_001.wav
   │   ├── sample_002.wav
   │   └── ...
   └── metadata.csv
   ```

2. metadata.csv format:
   ```csv
   audio_path,transcript
   audio/sample_001.wav,Kaixo mundua
   audio/sample_002.wav,Eskerrik asko
   ```

3. Upload the folder to Colab or mount Google Drive

In [ ]:
# Mount Google Drive (optional - if your dataset is on Drive)
from google.colab import drive
drive.mount('/content/drive')

# Set dataset path
# Example: DATASET_PATH = '/content/drive/MyDrive/euskera_data'
DATASET_PATH = '/content/euskera_data'  # Modify this path
METADATA_FILE = f'{DATASET_PATH}/metadata.csv'

### Validate Dataset

Check if your dataset is properly formatted:

In [ ]:
import os
import pandas as pd

# Check if metadata file exists
if not os.path.exists(METADATA_FILE):
    print(f"❌ Metadata file not found: {METADATA_FILE}")
    print("   Please upload your dataset first!")
else:
    # Load and validate metadata
    df = pd.read_csv(METADATA_FILE)
    print(f"✅ Found {len(df)} samples in metadata")
    
    # Check columns
    if 'audio_path' in df.columns and 'transcript' in df.columns:
        print("✅ Metadata format is correct")
        
        # Check some audio files
        missing = 0
        for i, row in df.head(10).iterrows():
            audio_path = os.path.join(DATASET_PATH, row['audio_path'])
            if not os.path.exists(audio_path):
                missing += 1
        
        if missing > 0:
            print(f"⚠️  {missing} audio files not found (checked first 10)")
        else:
            print("✅ Audio files are accessible")
        
        # Show sample
        print("\n📋 Sample entries:")
        print(df.head())
    else:
        print("❌ Metadata must have 'audio_path' and 'transcript' columns")

## 3. Configure Training

Set training parameters:

In [ ]:
# Training configuration
CONFIG = {
    # Model
    'base_model_name': 'nineninesix/kani-tts-400m-0.3-pt',
    'codec_model_name': 'nvidia/nemo-nano-codec-22khz-0.6kbps-12.5fps',
    
    # Dataset
    'dataset_path': DATASET_PATH,
    'transcripts_file': METADATA_FILE,
    
    # Output
    'output_dir': '/content/models/kani-tts-euskera',
    
    # Training hyperparameters
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,  # Reduce if OOM
    'gradient_accumulation_steps': 8,
    'learning_rate': 5e-5,
    'warmup_steps': 500,
    'max_seq_length': 1024,
    'fp16': True,
    'gradient_checkpointing': True,
}

print("✅ Configuration set:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

## 4. Download Finetuning Script

Get the finetuning script from the repository:

In [ ]:
# Clone the repository
!git clone https://github.com/maldalur/kani-tts.git

# Copy finetuning scripts to working directory
!cp kani-tts/finetuning/euskera/finetune_euskera.py .

print("✅ Finetuning script downloaded")

## 5. Start Training

This will take several hours depending on your dataset size.

**Expected times:**
- 100 samples: ~30 minutes
- 500 samples: ~2 hours
- 1000 samples: ~4 hours

You can monitor progress with TensorBoard in the next cell.

In [ ]:
# Start training
!python finetune_euskera.py \
    --base_model {CONFIG['base_model_name']} \
    --dataset_path {CONFIG['dataset_path']} \
    --transcripts_file {CONFIG['transcripts_file']} \
    --output_dir {CONFIG['output_dir']} \
    --num_epochs {CONFIG['num_train_epochs']} \
    --batch_size {CONFIG['per_device_train_batch_size']} \
    --learning_rate {CONFIG['learning_rate']}

In [ ]:
# Optional: Launch TensorBoard to monitor training
%load_ext tensorboard
%tensorboard --logdir {CONFIG['output_dir']}/logs

## 6. Test the Trained Model

Generate speech in Basque using your trained model:

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from nemo.collections.tts.models import AudioCodecModel
import soundfile as sf
from IPython.display import Audio

# Load the trained model
print("Loading trained model...")
model_path = CONFIG['output_dir']
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# Load codec
print("Loading codec...")
codec_model = AudioCodecModel.from_pretrained(CONFIG['codec_model_name'])
codec_model.eval()
if torch.cuda.is_available():
    codec_model = codec_model.cuda()

print("✅ Models loaded successfully!")

In [ ]:
# Generate speech
TEXT = "Kaixo, zer moduz zaude?"  # Change this to your Basque text

print(f"Generating speech for: {TEXT}")

# Tokenize
inputs = tokenizer(TEXT, return_tensors="pt", add_special_tokens=False).to(model.device)
start_token = torch.tensor([[1]], device=inputs['input_ids'].device)  # START_OF_TEXT
audio_start_token = torch.tensor([[64000]], device=inputs['input_ids'].device)  # START_OF_AUDIO
input_ids = torch.cat([start_token, inputs['input_ids'], audio_start_token], dim=1)

# Generate
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1000,
        temperature=0.6,
        top_p=0.95,
        do_sample=True,
    )

# Extract audio tokens
generated_ids = outputs[0].cpu().numpy()
audio_start_pos = np.where(generated_ids == 64000)[0]
if len(audio_start_pos) > 0:
    audio_start_pos = audio_start_pos[0] + 1
    end_pos = np.where(generated_ids == 64399)[0]
    audio_end_pos = end_pos[0] if len(end_pos) > 0 else len(generated_ids)
    
    audio_tokens = generated_ids[audio_start_pos:audio_end_pos]
    audio_tokens = audio_tokens[(audio_tokens >= 64001) & (audio_tokens <= 64399)]
    
    print(f"Generated {len(audio_tokens)} audio tokens")
    
    # Decode (simplified - you may need to adjust based on codec)
    # This is a placeholder - actual decoding depends on codec implementation
    print("⚠️  Audio decoding step needs codec-specific implementation")
    print("   See inference_euskera.py for complete implementation")
else:
    print("❌ No audio tokens generated")

## 7. Save and Download Model

Save your trained model:

In [ ]:
# Model is already saved in CONFIG['output_dir']
print(f"✅ Model saved at: {CONFIG['output_dir']}")
print("\nTo download:")
print("1. Find the folder in the Files panel (left sidebar)")
print("2. Right-click and select 'Download'")
print("\nOr zip and download:")
print(f"!zip -r euskera_model.zip {CONFIG['output_dir']}")

# Optional: Save to Google Drive
# !cp -r {CONFIG['output_dir']} /content/drive/MyDrive/

## 8. Next Steps

**Using Your Model:**
1. Download the model from Colab
2. Use the `inference_euskera.py` script locally
3. Or upload to Hugging Face Hub for easy sharing

**Improving Quality:**
- Collect more training data (500+ samples recommended)
- Train for more epochs (5-10)
- Ensure high-quality audio recordings
- Verify accurate transcriptions

**Resources:**
- [Full Documentation](https://github.com/maldalur/kani-tts/tree/main/finetuning/euskera)
- [Discord Community](https://discord.gg/NzP3rjB4SB)
- [Main Repository](https://github.com/maldalur/kani-tts)

---

**🎉 Congratulations! You've finetuned Kani TTS for Basque!**